# S30 - Unidad 2. Evidencia de Aprendizaje 3 - Aprendizaje supervisado y no supervisado

# Integrantes

Natalia Jadith Castro Ospino

Oscar Luis Maqruez Arrieta


# 0. Pregunta de negocio y construcción automática del DataFrame

# Pregunta de Negocio

¿Es posible identificar el estilo de consumo de los clientes teniendo en cuenta su ubicación geográfica y el valor de su cuota mensual?

# A quien beneficia?

Podriamos decir que la respuesta a esta pregunta beneficiaria a empresas que ofrecen algun servicio de suscripción o plataforma de netretenimiento. Esta tendria como objetivo tener en cuenta los datos de los clientes sobre su hubicacion geografica y dependiendo de su posicion organizar el valor de su cuota mensual de suscripción.

# Que desición apoya?

Permite conocer los diferentes estilos de consumo de los clientes y apoyar la creación de planes, promociones y estrategias de fidelización.

# Dataset seleccionado
   
- Nombre: Customer Profile.
  
- Identificador: hpestrellag/customer_profile.
  
- Plataforma: Hugging Face.
  
- Formato: CSV.
  
- Split: train.
  
- Cantidad de registros: 500.000.
  
- Licencia: CC BY-NC 4.0.
  
Enlace: https://huggingface.co/datasets/hpestrellag/customer_profile

# Columnas relevantes

Columna	 ------------->            Descripción

customer_id	   ------------>      Identificador del cliente.

department	 --------->         Departamento o subdivisión.

consumption_style	--------->  Estilo de consumo.

locality	------------>          Localidad del cliente.

province	----------->          Provincia del cliente.

monthly_fee_usd	--------->      Cuota mensual en dólares.


# ¿Por qué escogimos este dataset?

Escogimos este dataset porque contiene información sobre la ubicación de los clientes, el valor de su cuota mensual y sus diferentes estilos de consumo.

Estas características permiten plantear un problema de clasificación para identificar el estilo de consumo de los clientes.

# Importamos la clase de carga

In [17]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "clases").exists():
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

from clases.hf_loader import HFDatasetLoader

In [18]:
# Configuramos el dataset

loader = HFDatasetLoader(
    dataset_name="hpestrellag/customer_profile",
    split="train",
    filename="datos/hpestrellag/customer_profile.csv",
)

In [19]:
# Descargamos y cargamos

df = loader.load()
df.head(10)

,customer_id,department,consumption_style,locality,province,monthly_fee_usd
0,SLX83969,department_cat_153,heavy-user,locality_cat_163,Misiones,9.12
1,UTT27594,department_cat_1,occasional,locality_cat_268,Tucumán,10.15
2,HER11355,department_cat_1,occasional,locality_cat_102,Buenos Aires,18.93
3,ZRQ19908,department_cat_142,weekend-viewer,locality_cat_29,Santiago del Estero,7.36
4,EAT08840,department_cat_142,occasional,locality_cat_90,Córdoba,7.56
5,KYJ36841,department_cat_1,occasional,locality_cat_1,Santiago del Estero,8.31
6,FWX15967,department_cat_120,occasional,locality_cat_335,Santa Fe,8.07
7,YFL75558,department_cat_131,weekend-viewer,locality_cat_630,Buenos Aires,9.77
8,SYQ06673,department_cat_1,occasional,locality_cat_658,Córdoba,11.06
9,ULR53254,department_cat_112,weekend-viewer,locality_cat_445,Córdoba,11.55


# 1. Evidencia de preprocesamiento

In [20]:
# Tamaño del dataset

print("Tamaño del DataFrame:", df.shape)

Tamaño del DataFrame: (500000, 6)


In [21]:
# Tipo de datos
print("Tipos de datos:")

print(df.dtypes)

Tipos de datos:
customer_id              str
department               str
consumption_style        str
locality                 str
province                 str
monthly_fee_usd      float64
dtype: object


In [22]:
# Revisamos que valores nulos existes

print("\nValores nulos:")
print(df.isnull().sum())


Valores nulos:
customer_id          0
department           0
consumption_style    0
locality             0
province             0
monthly_fee_usd      0
dtype: int64


In [23]:
# En esta parte revisamos que registros estan duplicados

print("\nRegistros duplicados:", df.duplicated().sum())


Registros duplicados: 0


Revisamos que nuestro dataset esta completamente limpio desde el inicio. Se revisaron los valores nulos y los registros duplicados del dataset. Los resultados mostraron que no existen valores faltantes ni registros repetidos, por lo que no fue necesario aplicar operaciones de eliminación o imputación.

In [24]:
#  las variables predictoras
X = df[[
    "department",
    "locality",
    "province",
    "monthly_fee_usd"
]]

#variable objetivo
y = df["consumption_style"]

# Mostrar las variables
print("Variables predictoras:")
display(X.head())

print("\nVariable objetivo:")
display(y.head())

Variables predictoras:


,department,locality,province,monthly_fee_usd
0,department_cat_153,locality_cat_163,Misiones,9.12
1,department_cat_1,locality_cat_268,Tucumán,10.15
2,department_cat_1,locality_cat_102,Buenos Aires,18.93
3,department_cat_142,locality_cat_29,Santiago del Estero,7.36
4,department_cat_142,locality_cat_90,Córdoba,7.56



Variable objetivo:


0        heavy-user
1        occasional
2        occasional
3    weekend-viewer
4        occasional
Name: consumption_style, dtype: str

In [25]:
# Informacion General
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   customer_id        500000 non-null  str    
 1   department         500000 non-null  str    
 2   consumption_style  500000 non-null  str    
 3   locality           500000 non-null  str    
 4   province           500000 non-null  str    
 5   monthly_fee_usd    500000 non-null  float64
dtypes: float64(1), str(5)
memory usage: 51.7 MB


In [26]:
# Categorias

# categorías de consumo
print("Estilos de consumo:")
print(df["consumption_style"].value_counts())

# categorías de ubicación
print("\nDepartamentos:")
print(df["department"].nunique())

print("\nLocalidades:")
print(df["locality"].nunique())

print("\nProvincias:")
print(df["province"].nunique())

Estilos de consumo:
consumption_style
occasional        288181
casual             98581
weekend-viewer     73431
binge-watcher      26248
heavy-user         13559
Name: count, dtype: int64

Departamentos:
196

Localidades:
679

Provincias:
23


# Separamos datos de entrenamiestos y pruebas

In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


print("Datos de entrenamiento:", X_train.shape)
print("Datos de prueba:", X_test.shape)

Datos de entrenamiento: (400000, 4)
Datos de prueba: (100000, 4)


In [28]:
# Covertir categorias ya que random Forest trabaja con Valores numericos

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

#  columnas categóricas
columnas_categoricas = [
    "department",
    "locality",
    "province"
]

# preprarar transformación
preprocesador = ColumnTransformer(
    transformers=[
        ("categoricas", OneHotEncoder(handle_unknown="ignore"),
         columnas_categoricas)
    ],
    remainder="passthrough"
)

# Ajustar con entrenamiento y transformar
X_train_procesado = preprocesador.fit_transform(X_train)

# Transformar los datos de prueba
X_test_procesado = preprocesador.transform(X_test)

print("Entrenamiento procesado:", X_train_procesado.shape)
print("Prueba procesada:", X_test_procesado.shape)

Entrenamiento procesado: (400000, 891)
Prueba procesada: (100000, 891)


# 2. Selección y funcionamiento del modelo

## Modelo seleccionado


Random Forest

Tipo de aprendisaje: Supervisado



Se seleccionó el algoritmo Random Forest porque permite clasificar a los clientes según su estilo de consumo, utilizando características como su 

ubicación geográfica y el valor de su cuota mensual.

## Tipo de aprendizaje

Nuestro proyecto utiliza aprendizaje supervisado porque contamos con una variable que ya contiene la respuesta que queremos predecir.

El modelo aprenderá a partir de los datos de los clientes y utilizará esa información para predecir el estilo de consumo de nuevos registros.

## Variables del modelo

Utilizaremos las siguientes variables:

#### Tipo-----------------Variable----------------------Descripción

Predictora (X)------------department--------------------Departamento del cliente.

Predictora (X)------------locality----------------------Localidad del cliente.

Predictora (X)------------province----------------------Provincia del cliente.

Predictora (X)------------monthly_fee_usd---------------Cuota mensual en dólares.

Objetivo (Y)--------------consumption_style-------------Estilo de consumo del cliente.

La variable objetivo es consumption_style, porque representa la categoría que queremos que el modelo aprenda a identificar.

Para conocer las clases reales que utilizará el modelo, ejecuta esta celda:

In [29]:
# Mostrar las clases de la variable objetivo
print("Clases de consumo:")
print(y.value_counts())

print("\nCantidad de clases:")
print(y.nunique())

Clases de consumo:
consumption_style
occasional        288181
casual             98581
weekend-viewer     73431
binge-watcher      26248
heavy-user         13559
Name: count, dtype: int64

Cantidad de clases:
5
